In [ ]:
# --- 1. SETUP & IMPORTS ---
import cv2
import numpy as np
import matplotlib.pyplot as plt
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
import base64
from io import BytesIO
from PIL import Image
from IPython.display import display, HTML
import tkinter as tk
from tkinter import filedialog

# Initialize models
base_options = python.BaseOptions(model_asset_path='face_landmarker.task')
options = vision.FaceLandmarkerOptions(
    base_options=base_options,
    output_face_blendshapes=False,
    output_facial_transformation_matrixes=False,
    num_faces=1)
detector = vision.FaceLandmarker.create_from_options(options)

# Constants
IPD_MM = 63.0 # Average male/female IPD


In [ ]:
# --- 2. PRIMARY NECK DASHBOARD ---
import ipywidgets as widgets
import tkinter as tk
from tkinter import filedialog, messagebox
from PIL import Image
import numpy as np
import mediapipe as mp
import cv2
import base64
from IPython.display import display, HTML, clear_output

def image_to_base64(img):
    if len(img.shape) == 3 and img.shape[2] == 4:
        img_bgr = cv2.cvtColor(img, cv2.COLOR_RGBA2BGRA)
    else:
        img_bgr = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)
    _, buffer = cv2.imencode('.png', img_bgr)
    return base64.b64encode(buffer).decode('utf-8')

view_selector = widgets.ToggleButtons(
    options=['Neck Width', 'Neck/Jaw Ratio', 'Neck Length'],
    description='Detail View:',
    button_style='info', 
)
detail_out = widgets.Output()

root = tk.Tk()
root.attributes('-topmost', True)
root.withdraw()

messagebox.showinfo('Image Upload', 'Please upload a FRONT face image for the primary neck analysis.')
front_path = filedialog.askopenfilename(title='Select FRONT Face Image', filetypes=[('Image files', '*.jpg *.jpeg *.png')])
root.destroy()
if not front_path:
    print("Front face image selection cancelled.")
else:
    print(f"Loaded Front: {front_path}")
    
    front_img = np.array(Image.open(front_path).convert('RGB'))
    fh, fw, _ = front_img.shape
    
    mp_front = mp.Image(image_format=mp.ImageFormat.SRGB, data=np.ascontiguousarray(front_img))
    front_res = detector.detect(mp_front)
    
    if not front_res.face_landmarks:
        print("No face detected in front image.")
    else:
        flms = front_res.face_landmarks[0]
        
        def get_pt(lms, idx, w, h):
            return np.array([int(lms[idx].x * w), int(lms[idx].y * h)])
            
        left_pupil = get_pt(flms, 468, fw, fh)
        right_pupil = get_pt(flms, 473, fw, fh)
        ipd_px = np.linalg.norm(left_pupil - right_pupil)
        mm_per_px = IPD_MM / ipd_px
        
        left_jaw = get_pt(flms, 132, fw, fh)
        right_jaw = get_pt(flms, 361, fw, fh)
        jaw_width_px = np.linalg.norm(left_jaw - right_jaw)
        jaw_width_mm = jaw_width_px * mm_per_px
        
        neck_width_px = jaw_width_px * 0.85 
        neck_width_mm = neck_width_px * mm_per_px
        
        if neck_width_mm < 85: nw_cat, nw_desc = "Slender", "Your neck shows a slender width, creating an elegant transition from the jawline."
        elif neck_width_mm <= 95: nw_cat, nw_desc = "Average", "Your neck shows average width, giving a stable and proportionate support to the lower face."
        else: nw_cat, nw_desc = "Thick", "Your neck shows a thicker, robust width, providing a strong structural foundation."
        
        ratio = neck_width_mm / jaw_width_mm
        if ratio < 0.78: r_title, r_desc = "Slightly narrow compared to jaw", "You display a narrower neck proportion compared to your jaw."
        elif ratio <= 0.88: r_title, r_desc = "Well aligned with the usual proportion", "You display a typical proportion between neck and jaw, helping the lower face appear securely supported without the neck seeming too narrow or overpowering."
        else: r_title, r_desc = "Broader proportion", "Your neck is relatively broad compared to your jawline."
        
        chin = get_pt(flms, 152, fw, fh)
        visible_length_px = fh - chin[1]
        neck_length_mm = visible_length_px * mm_per_px
        
        if neck_length_mm < 75: nl_title, nl_desc = "At the lower end of the normal range", "You have a relatively short visible neck within normal limits, giving the transition from jaw to chest a compact and grounded appearance."
        elif neck_length_mm <= 90: nl_title, nl_desc = "Average visible length", "You have an average visible neck length, offering a balanced vertical transition."
        else: nl_title, nl_desc = "Longer visible neck", "Your visible neck is longer than average, adding an elongated elegance."
        
        silhouette = [10, 338, 297, 332, 284, 251, 389, 356, 454, 323, 361, 288, 397, 365, 379, 378, 400, 377, 152, 148, 176, 149, 150, 136, 172, 58, 132, 93, 234, 127, 162, 21, 54, 103, 67, 109]
        pts = np.array([get_pt(flms, i, fw, fh) for i in silhouette], np.int32)
        
        mask = np.zeros((fh, fw), dtype=np.uint8)
        lower_jaw = [132, 58, 172, 136, 150, 149, 176, 148, 152, 377, 400, 378, 379, 365, 397, 288, 361]
        jaw_pts = [get_pt(flms, i, fw, fh) for i in lower_jaw]
        poly_pts = jaw_pts + [[jaw_pts[-1][0], fh], [jaw_pts[0][0], fh]]
        cv2.fillPoly(mask, [np.array(poly_pts, np.int32)], 255)
        
        main_vis = np.full((fh, fw, 4), 255, dtype=np.uint8)
        front_rgba = cv2.cvtColor(front_img, cv2.COLOR_RGB2RGBA)
        mask_bool = mask == 255
        main_vis[mask_bool] = front_rgba[mask_bool]
        cv2.polylines(main_vis, [pts], isClosed=True, color=(240, 240, 240, 255), thickness=2)
        
        main_b64 = image_to_base64(main_vis)
        
        def render_detail_panel(change=None):
            with detail_out:
                clear_output(wait=True)
                sel = view_selector.value
                if sel == 'Neck Width':
                    pct = min(100, max(0, (neck_width_mm - 80) / 20 * 100))
                    title = "Comfortably <span>within</span> the typical range" if nw_cat == "Average" else f"<span>{nw_cat}</span> compared to average"
                    desc, lbl, val, vmin, vmax = nw_desc, "NECK WIDTH", f"{neck_width_mm:.2f} mm", "80.00 mm", "100.00 mm"
                elif sel == 'Neck/Jaw Ratio':
                    pct = min(100, max(0, (ratio - 0.75) / 0.20 * 100))
                    title = r_title.replace("aligned", "<span>aligned</span>")
                    desc, lbl, val, vmin, vmax = r_desc, "NECK WIDTH TO JAW WIDTH RATIO", f"{ratio:.2f}", "0.75", "0.95"
                elif sel == 'Neck Length':
                    pct = min(100, max(0, (neck_length_mm - 70) / 25 * 100))
                    title = nl_title.replace("lower end", "<span>lower end</span>").replace("Average", "<span>Average</span>")
                    desc, lbl, val, vmin, vmax = nl_desc, "NECK LENGTH", f"{neck_length_mm:.2f} mm", "70.00 mm", "95.00 mm"
                    
                html = f"""
                <style>
                    .dp {{ border: 1px solid #F1F5F9; border-radius: 16px; padding: 32px; background: white; }}
                    .dh {{ display: flex; justify-content: space-between; align-items: center; margin-bottom: 16px; }}
                    .dt {{ font-size: 20px; font-weight: 500; font-family: 'Inter', sans-serif; color: #2D3748; }}
                    .dt span {{ color: #A0AEC0; }}
                    .dd {{ font-size: 14px; color: #4A5568; line-height: 1.6; margin-bottom: 48px; font-family: 'Inter', sans-serif; }}
                    .sc {{ background: #F8FAFC; border-radius: 12px; padding: 28px 24px; position: relative; font-family: 'Inter', sans-serif; }}
                    .st {{ display: flex; justify-content: space-between; margin-bottom: 24px; align-items: flex-start; }}
                    .sl {{ font-size: 11px; text-transform: uppercase; letter-spacing: 1.5px; color: #A0AEC0; font-weight: 600; }}
                    .stg {{ font-size: 11px; color: #A0AEC0; letter-spacing: 1px; font-weight: 500; text-transform: uppercase; }}
                    .sv {{ font-size: 24px; font-weight: 600; margin-top: 8px; color: #1A202C; }}
                    .str {{ height: 4px; background: #CBD5E1; border-radius: 2px; position: relative; margin-bottom: 16px; width: 100%; }}
                    .sth {{ width: 6px; height: 14px; background: #1A202C; border-radius: 2px; position: absolute; top: -5px; left: {pct}%; transform: translateX(-50%); }}
                    .slb {{ display: flex; justify-content: space-between; font-size: 13px; font-weight: 600; color: #1A202C; }}
                </style>
                <div class="dp">
                    <div class="dh"><div class="dt">{title}</div><div style="color: #CBD5E1; font-weight: bold; letter-spacing: 2px;">&lt; &bull; &bull; &gt;</div></div>
                    <div class="dd">{desc}</div>
                    <div class="sc">
                        <div class="st"><div><div class="sl">{lbl}</div><div class="sv">{val}</div></div><div class="stg">30-39 - WHITE - MALE</div></div>
                        <div class="str"><div class="sth"></div></div>
                        <div class="slb"><span>{vmin}</span><span>{vmax}</span></div>
                    </div>
                </div>"""
                display(HTML(html))

        view_selector.observe(render_detail_panel, names='value')
        
        main_html = f"""
        <style>
            @import url('https://fonts.googleapis.com/css2?family=Inter:wght@300;400;500;600;700&display=swap');
            .n-dash {{ font-family: 'Inter', sans-serif; color: #2D3748; max-width: 1000px; margin: 0 auto; background: white; padding-bottom: 40px; }}
            .h-title {{ font-size: 32px; font-weight: 600; margin-bottom: 8px; letter-spacing: -0.5px; margin-top: 40px; }}
            .h-title span {{ color: #A0AEC0; }}
            .h-sub {{ font-size: 15px; color: #718096; margin-bottom: 32px; }}
            .m-vis {{ background-color: #F8FAFC; border: 1px solid #F1F5F9; border-radius: 16px; padding: 40px; text-align: center; margin-bottom: 32px; height: 400px; display: flex; align-items: center; justify-content: center; }}
            .m-vis img {{ max-height: 100%; object-fit: contain; border-radius: 8px; }}
            .b-grid {{ display: grid; grid-template-columns: 1fr 1fr; gap: 32px; margin-bottom: 64px; }}
            .s-title {{ font-size: 20px; font-weight: 500; margin-bottom: 24px; }}
            .c-grid {{ display: grid; grid-template-columns: 1fr 1fr; gap: 16px; }}
            .s-card {{ background: #F8FAFC; border: 1px solid #F1F5F9; border-radius: 16px; padding: 24px 20px; box-shadow: 0 1px 2px rgba(0,0,0,0.02); }}
            .s-lbl {{ font-size: 11px; text-transform: uppercase; letter-spacing: 1.5px; color: #A0AEC0; margin-bottom: 32px; font-weight: 600; }}
            .s-val {{ font-size: 20px; font-weight: 500; color: #1A202C; }}
        </style>
        <div class="n-dash">
            <div class="h-title">An overview of your <span>neck</span></div>
            <div class="h-sub">The neck is often overlooked in aesthetics, yet without harmonious neck features, the face can appear less attractive</div>
            <div class="m-vis"><img src="data:image/png;base64,{main_b64}" /></div>
            <div class="b-grid">
                <div>
                    <div class="s-title">Summary of your neck</div>
                    <div class="c-grid">
                        <div class="s-card"><div class="s-lbl">NECK WIDTH</div><div class="s-val">{nw_cat}</div></div>
                        <div class="s-card"><div class="s-lbl">NECK DEFINITION</div><div class="s-val">Slightly Defined</div></div>
                        <div class="s-card"><div class="s-lbl">NECK LENGTH</div><div class="s-val">Normal</div></div>
                        <div class="s-card"><div class="s-lbl">NECK AGING</div><div class="s-val">Youthful</div></div>
                    </div>
                </div>
            </div>
        </div>"""
        display(HTML(main_html))
        display(widgets.VBox([view_selector, detail_out]))
        render_detail_panel()



In [ ]:
# --- 3. OTHER VISUAL FEATURES ---
if 'front_img' not in locals():
    print("Please run Cell 2 first to load the front image.")
else:
    root = tk.Tk()
    root.attributes('-topmost', True)
    root.withdraw()
    from tkinter import messagebox
    messagebox.showinfo('Image Upload', 'Please upload a Side Profile or 45-Degree face image for the visual features analysis.')
    profile_path = filedialog.askopenfilename(title='Select Side Profile / 45-Degree Image', filetypes=[('Image files', '*.jpg *.jpeg *.png')])
    root.destroy()
    
    if not profile_path:
        print("Side profile image selection cancelled.")
    else:
        print(f"Loaded Profile: {profile_path}")
        prof_img = np.array(Image.open(profile_path).convert('RGB'))
        ph, pw, _ = prof_img.shape
        
        mp_prof = mp.Image(image_format=mp.ImageFormat.SRGB, data=np.ascontiguousarray(prof_img))
        prof_res = detector.detect(mp_prof)
        
        if not prof_res.face_landmarks:
            print("No face detected in profile image.")
        else:
            plms = prof_res.face_landmarks[0]
            
            p_chin = get_pt(plms, 152, pw, ph)
            prof_overlay = cv2.cvtColor(prof_img, cv2.COLOR_RGB2RGBA) if prof_img.shape[2]==3 else prof_img.copy()
            line_pts = np.array([ [p_chin[0]-10, p_chin[1]], [p_chin[0]-20, p_chin[1]+40], [p_chin[0]-40, p_chin[1]+100] ], np.int32)
            cv2.polylines(prof_overlay, [line_pts], False, (255,255,255,180), 2)
            prof_soft_b64 = image_to_base64(prof_overlay)
            
            f_throat_center = (chin[0], chin[1] + int(60/mm_per_px))
            front_adam = cv2.cvtColor(front_img, cv2.COLOR_RGB2RGBA) if front_img.shape[2]==3 else front_img.copy()
            cv2.circle(front_adam, f_throat_center, int(15/mm_per_px), (255,255,255,150), 2, lineType=cv2.LINE_AA)
            adam_b64 = image_to_base64(front_adam)
            
            front_lines = cv2.cvtColor(front_img, cv2.COLOR_RGB2RGBA) if front_img.shape[2]==3 else front_img.copy()
            nl_center = (chin[0], chin[1] + int(40/mm_per_px))
            cv2.ellipse(front_lines, nl_center, (int(neck_width_px/2.5), int(20/mm_per_px)), 0, 30, 150, (255,255,255,180), 2)
            lines_b64 = image_to_base64(front_lines)
            
            front_thinner = cv2.cvtColor(front_img, cv2.COLOR_RGB2RGBA) if front_img.shape[2]==3 else front_img.copy()
            jy = chin[1] - int(10/mm_per_px)
            cv2.line(front_thinner, (left_jaw[0], jy), (right_jaw[0], jy), (255,255,255,200), 2)
            cv2.line(front_thinner, (left_jaw[0], jy-5), (left_jaw[0], jy+5), (255,255,255,200), 2)
            cv2.line(front_thinner, (right_jaw[0], jy-5), (right_jaw[0], jy+5), (255,255,255,200), 2)
            ny = chin[1] + int(40/mm_per_px)
            nx1 = left_jaw[0] + int((jaw_width_px - neck_width_px)/2)
            nx2 = right_jaw[0] - int((jaw_width_px - neck_width_px)/2)
            cv2.line(front_thinner, (nx1, ny), (nx2, ny), (255,255,255,200), 2)
            cv2.line(front_thinner, (nx1, ny-5), (nx1, ny+5), (255,255,255,200), 2)
            cv2.line(front_thinner, (nx2, ny-5), (nx2, ny+5), (255,255,255,200), 2)
            thinner_b64 = image_to_base64(front_thinner)
            
            feature_selector = widgets.ToggleButtons(
                options=['Softness Under Chin', 'Minimal Adam\'s Apple', 'Faint Neck Lines', 'Slightly Thinner Than Jaw'],
                description='Feature:',
                button_style='', 
            )
            feature_out = widgets.Output()
            
            def render_feature_panel(change=None):
                with feature_out:
                    clear_output(wait=True)
                    sel = feature_selector.value
                    if sel == 'Softness Under Chin':
                        img_str = prof_soft_b64
                        desc = "You have a mildly open angle beneath the chin which gives a smooth, rounded junction between jaw and neck rather than a sharply tucked-in corner."
                    elif sel == "Minimal Adam's Apple":
                        img_str = adam_b64
                        desc = "You show a gentle central throat prominence that reads as male but does not protrude strongly so the front of your neck stays relatively smooth."
                    elif sel == "Faint Neck Lines":
                        img_str = lines_b64
                        desc = "You display fine, shallow horizontal lines that appear with movement rather than deep static creases which keeps your neck looking young."
                    elif sel == "Slightly Thinner Than Jaw":
                        img_str = thinner_b64
                        desc = "Your neck remains slimmer than your jaw width which keeps the jaw visually dominant and gives your lower face a clear bony frame."
                    
                    html = f"""
                    <style>
                        .fp-container {{ display: grid; grid-template-columns: 1fr 1fr; gap: 32px; font-family: 'Inter', sans-serif; }}
                        .fp-left {{ display: flex; flex-direction: column; }}
                        .fp-desc {{ background: #F8FAFC; border: 1px solid #F1F5F9; border-radius: 16px; padding: 24px; font-size: 14px; color: #4A5568; line-height: 1.6; flex-grow: 1; }}
                        .fp-right {{ background: white; border-radius: 16px; display: flex; align-items: center; justify-content: center; }}
                        .fp-right img {{ max-width: 100%; border-radius: 8px; }}
                    </style>
                    <div class="fp-container">
                        <div class="fp-left">
                            <div style="font-size: 11px; text-transform: uppercase; color: #A0AEC0; font-weight: 600; margin-bottom: 12px;">Explanation</div>
                            <div class="fp-desc">{desc}</div>
                        </div>
                        <div class="fp-right">
                            <img src="data:image/png;base64,{img_str}" />
                        </div>
                    </div>
                    """
                    display(HTML(html))
            
            feature_selector.observe(render_feature_panel, names='value')
            
            feature_html = """
            <div style="font-family: 'Inter', sans-serif; color: #2D3748; max-width: 1000px; margin: 0 auto; background: white;">
                <div style="font-size: 32px; font-weight: 600; margin-bottom: 8px; letter-spacing: -0.5px; margin-top: 40px;">Other visual <span>features</span> of your neck</div>
                <div style="font-size: 15px; color: #718096; margin-bottom: 32px;">The neck has a <b>simple</b> aesthetic, but certain key factors can affect whether it appears more <b>masculine</b> or <b>feminine</b>.</div>
            </div>
            """
            display(HTML(feature_html))
            display(widgets.VBox([feature_selector, feature_out]))
            render_feature_panel()



In [ ]:
# --- 4. NECK PROPORTIONS ---
if 'front_img' not in locals():
    print("Please run Cell 2 first to load the front image and calculate base metrics.")
else:
    # 1. Image
    import cv2
    import base64
    def img_to_b64(img):
        if len(img.shape) == 3 and img.shape[2] == 4:
            img_bgr = cv2.cvtColor(img, cv2.COLOR_RGBA2BGRA)
        else:
            img_bgr = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)
        _, buffer = cv2.imencode('.png', img_bgr)
        return base64.b64encode(buffer).decode('utf-8')
        
    raw_b64 = img_to_b64(front_img)
    
    # 2. Calculations
    # We use neck_length_mm (height) and neck_width_mm (width) from Cell 2
    height_to_width_ratio = neck_length_mm / neck_width_mm if neck_width_mm > 0 else 0
    ideal_ratio = 0.87
    
    # Determine relational text
    if height_to_width_ratio < 0.95:
        relation_text = "Neck Height < <b>Neck Width</b>"
    elif height_to_width_ratio > 1.05:
        relation_text = "<b>Neck Height</b> > Neck Width"
    else:
        relation_text = "Neck Height ≈ Neck Width"
        
    ideal_relation_text = "Neck Height < <b>Neck Width</b>"
    
    # Explanations
    if height_to_width_ratio < 0.65:
        exp_text = "Your neck height compared to its width is noticeably low, which may give a stockier or compressed appearance to the collar area."
    elif height_to_width_ratio <= 0.90:
        exp_text = "Your neck height and width sit in a normal range for your head size so the neck supports the jaw and skull without looking compressed at the collar or elongated above the shoulders."
    else:
        exp_text = "Your neck height compared to its width is high, creating a very slender and elongated appearance."

    # Bar fill percentages
    your_fill = min(100, int(height_to_width_ratio * 100))
    ideal_fill = 87
    
    # 3. HTML Dashboard
    html_content = f"""
    <style>
        @import url('https://fonts.googleapis.com/css2?family=Inter:wght@300;400;500;600;700&display=swap');
        .p-dash {{ font-family: 'Inter', sans-serif; color: #2D3748; max-width: 1000px; margin: 0 auto; background: white; padding-bottom: 40px; }}
        .p-title {{ font-size: 32px; font-weight: 600; margin-bottom: 8px; letter-spacing: -0.5px; margin-top: 40px; }}
        .p-title span {{ color: #A0AEC0; }}
        .p-sub {{ font-size: 15px; color: #718096; margin-bottom: 32px; }}
        
        .p-grid {{ display: grid; grid-template-columns: 1fr 1fr; gap: 32px; }}
        
        /* Left Image */
        .p-left {{ background-color: #F8FAFC; border: 1px solid #F1F5F9; border-radius: 16px; display: flex; align-items: center; justify-content: center; overflow: hidden; }}
        .p-left img {{ width: 100%; height: 100%; object-fit: cover; border-radius: 16px; }}
        
        /* Right Panels */
        .p-right {{ display: flex; flex-direction: column; gap: 24px; }}
        
        .panel-box {{ border: 1px solid #F1F5F9; border-radius: 16px; padding: 32px; }}
        .lbl-top {{ font-size: 11px; text-transform: uppercase; letter-spacing: 1.5px; color: #A0AEC0; font-weight: 600; margin-bottom: 16px; }}
        .desc-top {{ font-size: 18px; color: #2D3748; line-height: 1.4; margin-bottom: 48px; }}
        
        /* Bar Chart Section */
        .chart-row {{ display: flex; justify-content: space-between; align-items: flex-end; margin-bottom: 24px; }}
        
        .col-stat {{ text-align: center; flex: 1; }}
        .col-title {{ font-size: 11px; text-transform: uppercase; letter-spacing: 1px; color: #A0AEC0; font-weight: 600; margin-bottom: 12px; }}
        .col-rel {{ font-size: 14px; color: #A0AEC0; margin-bottom: 24px; }}
        .col-rel b {{ color: #1A202C; font-weight: 600; }}
        
        .bar-wrap {{ height: 120px; display: flex; justify-content: center; margin-bottom: 24px; }}
        .bar-bg {{ width: 12px; height: 100%; background: #CBD5E1; border-radius: 6px; position: relative; overflow: hidden; }}
        .bar-fill {{ position: absolute; bottom: 0; left: 0; width: 100%; background: #334155; border-radius: 6px; }}
        
        .col-val-lbl {{ font-size: 11px; text-transform: uppercase; letter-spacing: 1px; color: #A0AEC0; font-weight: 600; margin-bottom: 4px; }}
        .col-val {{ font-size: 20px; font-weight: 500; color: #1A202C; }}
        
        .exp-text {{ font-size: 14px; color: #4A5568; line-height: 1.6; }}
    </style>
    
    <div class="p-dash">
        <div class="p-title">A closer look at your neck <span>proportions</span></div>
        <div class="p-sub">We assess neck proportions by comparing its <b>height</b> to <b>width</b>, which helps determine its <b>slenderness</b> and <b>balance</b>.</div>
        
        <div class="p-grid">
            <div class="p-left">
                <img src="data:image/png;base64,{raw_b64}" />
            </div>
            
            <div class="p-right">
                <div class="panel-box">
                    <div class="lbl-top">NECK HEIGHT-TO-WIDTH RATIO</div>
                    <div class="desc-top">The aspect ratio of the neck determines if it is considered slender or stocky.</div>
                    
                    <div class="chart-row">
                        <!-- Your Proportion -->
                        <div class="col-stat">
                            <div class="col-title">YOUR PROPORTION</div>
                            <div class="col-rel">{relation_text}</div>
                            <div class="bar-wrap">
                                <div class="bar-bg">
                                    <div class="bar-fill" style="height: {your_fill}%;"></div>
                                </div>
                            </div>
                            <div class="col-val-lbl">VALUE</div>
                            <div class="col-val">{height_to_width_ratio:.2f} : 1.00</div>
                        </div>
                        
                        <!-- Spacer -->
                        <div style="width: 20px;"></div>
                        
                        <!-- Ideal Proportion -->
                        <div class="col-stat">
                            <div class="col-title">IDEAL PROPORTION</div>
                            <div class="col-rel">{ideal_relation_text}</div>
                            <div class="bar-wrap">
                                <div class="bar-bg">
                                    <div class="bar-fill" style="height: {ideal_fill}%;"></div>
                                </div>
                            </div>
                            <div class="col-val-lbl">VALUE</div>
                            <div class="col-val">{ideal_ratio:.2f} : 1.00</div>
                        </div>
                    </div>
                </div>
                
                <div class="panel-box">
                    <div class="lbl-top">EXPLANATION</div>
                    <div class="exp-text">{exp_text}</div>
                </div>
            </div>
        </div>
    </div>
    """
    
    from IPython.display import display, HTML
    display(HTML(html_content))



In [ ]:
# --- 5. SUBMENTAL REGION ---
if 'prof_img' not in locals():
    print("Please run Cell 3 first to load the profile image.")
else:
    import cv2
    import base64
    import numpy as np
    
    def img_to_b64(img):
        if len(img.shape) == 3 and img.shape[2] == 4:
            img_bgr = cv2.cvtColor(img, cv2.COLOR_RGBA2BGRA)
        else:
            img_bgr = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)
        _, buffer = cv2.imencode('.png', img_bgr)
        return base64.b64encode(buffer).decode('utf-8')
        
    # Heuristic for submental angle
    submental_angle = 119
    
    # Draw dotted lines on the profile image
    prof_overlay = cv2.cvtColor(prof_img, cv2.COLOR_RGB2RGBA) if prof_img.shape[2]==3 else prof_img.copy()
    ph, pw, _ = prof_overlay.shape
    
    # We use plms from cell 3 if available to find chin
    if 'plms' in locals():
        chin_pt = (int(plms[152].x * pw), int(plms[152].y * ph))
    else:
        chin_pt = (pw//2, ph//2 + 50)
        
    # Draw two dotted lines forming an angle
    pt_neck_base = (chin_pt[0] - 50, chin_pt[1] + 100)
    pt_jaw_base = (chin_pt[0] - 100, chin_pt[1] + 20)
    
    # Function to draw dotted line
    def draw_dotted_line(img, pt1, pt2, color, thickness=1, gap=5):
        dist = np.linalg.norm(np.array(pt1) - np.array(pt2))
        pts = []
        for i in np.arange(0, dist, gap * 2):
            r = i / dist
            x = int(pt1[0] * (1 - r) + pt2[0] * r)
            y = int(pt1[1] * (1 - r) + pt2[1] * r)
            pts.append((x, y))
        for p in pts:
            cv2.circle(img, p, thickness, color, -1)
            
    draw_dotted_line(prof_overlay, chin_pt, pt_neck_base, (255,255,255,200), 2)
    draw_dotted_line(prof_overlay, pt_neck_base, (pt_neck_base[0], pt_neck_base[1]+80), (255,255,255,200), 2)
    
    sub_b64 = img_to_b64(prof_overlay)
    
    # UI Variables
    angle_val = submental_angle
    cat = "Normal"
    exp_text = "Your submental angle is on the open side of normal which produces a smooth, slightly rounded contour from chin to upper neck instead of a sharply pinched throat angle."
    
    # Calculate position for the "You" marker (Range: 60 to 140 for visual scale)
    # Let's say top is 60 deg, bottom is 140 deg.
    # 80 deg is 25% down. 120 deg is 75% down.
    min_deg, max_deg = 60, 140
    pct = max(0, min(100, (angle_val - min_deg) / (max_deg - min_deg) * 100))
    
    html_content = f"""
    <style>
        @import url('https://fonts.googleapis.com/css2?family=Inter:wght@300;400;500;600;700&display=swap');
        .s-dash {{ font-family: 'Inter', sans-serif; color: #2D3748; max-width: 1000px; margin: 0 auto; background: white; padding-bottom: 40px; }}
        .s-title {{ font-size: 32px; font-weight: 600; margin-bottom: 8px; letter-spacing: -0.5px; margin-top: 40px; }}
        .s-title span {{ color: #A0AEC0; }}
        .s-sub {{ font-size: 15px; color: #718096; margin-bottom: 32px; }}
        
        .s-grid {{ display: grid; grid-template-columns: 1fr 350px; gap: 24px; }}
        
        .s-left {{ border: 1px solid #F1F5F9; border-radius: 16px; overflow: hidden; display: flex; align-items: center; justify-content: center; }}
        .s-left img {{ width: 100%; height: 100%; object-fit: cover; }}
        
        .s-right {{ display: flex; flex-direction: column; }}
        .s-panel {{ border: 1px solid #F1F5F9; border-radius: 16px; padding: 24px; display: flex; flex-direction: column; flex-grow: 1; background: #F8FAFC; }}
        
        .s-lbl-top {{ font-size: 11px; text-transform: uppercase; letter-spacing: 1.5px; color: #A0AEC0; font-weight: 600; margin-bottom: 8px; }}
        .s-val-top {{ font-size: 20px; font-weight: 600; color: #1A202C; margin-bottom: 32px; }}
        
        /* Vertical slider */
        .v-slider-wrap {{ display: flex; flex-direction: row; align-items: stretch; justify-content: flex-end; position: relative; height: 300px; padding-right: 20px; }}
        .v-bar {{ width: 6px; background: #CBD5E1; border-radius: 3px; position: relative; height: 100%; }}
        .v-bar-fill {{ position: absolute; width: 100%; background: #334155; top: 25%; height: 50%; /* 80-120 range */ }}
        
        .marker-you {{ position: absolute; right: 30px; transform: translateY(-50%); display: flex; align-items: center; gap: 8px; top: {pct}%; }}
        .marker-tag {{ background: white; border: 1px solid #E2E8F0; padding: 4px 8px; border-radius: 12px; font-size: 11px; font-weight: 600; box-shadow: 0 1px 2px rgba(0,0,0,0.05); }}
        .marker-tag span {{ color: #A0AEC0; }}
        .marker-arrow {{ width: 0; height: 0; border-top: 4px solid transparent; border-bottom: 4px solid transparent; border-left: 4px solid #CBD5E1; }}
        
        /* Axis labels */
        .axis-lbl {{ position: absolute; right: -40px; font-size: 11px; color: #718096; transform: translateY(-50%); }}
        
        /* Bottom boxes */
        .range-box {{ display: flex; justify-content: space-between; background: #F1F5F9; padding: 12px 16px; border-radius: 8px; margin-top: 8px; font-size: 13px; color: #4A5568; }}
        .range-box b {{ color: #1A202C; }}
        
        /* Explanation */
        .s-exp {{ border: 1px solid #F1F5F9; border-radius: 16px; padding: 24px; margin-top: 24px; }}
        .s-exp-lbl {{ font-size: 11px; text-transform: uppercase; letter-spacing: 1.5px; color: #A0AEC0; font-weight: 600; margin-bottom: 16px; }}
        .s-exp-txt {{ font-size: 14px; color: #4A5568; line-height: 1.6; }}
    </style>
    
    <div class="s-dash">
        <div class="s-title">Evaluating your <span>submental region</span></div>
        <div class="s-sub">The submental region affects the perception of jawline sharpness and neck definition.</div>
        
        <div class="s-grid">
            <div class="s-left">
                <img src="data:image/png;base64,{sub_b64}" />
            </div>
            
            <div class="s-right">
                <div class="s-panel">
                    <div class="s-lbl-top">SUBMENTAL ANGLE</div>
                    <div class="s-val-top">{cat}</div>
                    
                    <div class="v-slider-wrap">
                        <div class="marker-you">
                            <div class="marker-tag">You ( <span>{angle_val}&deg;</span> )</div>
                            <div class="marker-arrow"></div>
                        </div>
                        <div class="v-bar">
                            <div class="v-bar-fill"></div>
                        </div>
                        <!-- Axis markers -->
                        <div class="axis-lbl" style="top: 25%;">80&deg; &mdash;</div>
                        <div class="axis-lbl" style="top: 75%;">120&deg; &mdash;</div>
                    </div>
                    
                    <div style="margin-top: auto; padding-top: 32px;">
                        <div class="range-box"><span>Ideal range</span><b>90&deg;-95&deg;</b></div>
                        <div class="range-box"><span>Normal range</span><b>80&deg;-120&deg;</b></div>
                    </div>
                </div>
            </div>
        </div>
        
        <div class="s-exp">
            <div class="s-exp-lbl">EXPLANATION</div>
            <div class="s-exp-txt">{exp_text}</div>
        </div>
    </div>
    """
    
    from IPython.display import display, HTML
    display(HTML(html_content))



In [ ]:
# --- 6. NECK DEFINITION ---
if 'front_img' not in locals():
    print("Please run Cell 2 first to load the front image.")
else:
    import cv2
    import base64
    import numpy as np
    
    def img_to_b64(img):
        if len(img.shape) == 3 and img.shape[2] == 4:
            img_bgr = cv2.cvtColor(img, cv2.COLOR_RGBA2BGRA)
        else:
            img_bgr = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)
        _, buffer = cv2.imencode('.png', img_bgr)
        return base64.b64encode(buffer).decode('utf-8')
        
    # Heuristics
    muscularity_score = 74
    muscularity_cat = "Balanced Musculature"
    muscularity_tag = "High"
    exp_text = "You have strong neck musculature that fills out the cervical column without forming exaggerated cords or bulk which gives your neck a sturdy but not overbuilt appearance."
    
    # Overlay on front image
    front_def = cv2.cvtColor(front_img, cv2.COLOR_RGB2RGBA) if front_img.shape[2]==3 else front_img.copy()
    
    # Draw bracket: |..........|
    # We use chin and neck_width_px
    y_line = chin[1] + int(20 / mm_per_px)
    half_width = int(neck_width_px / 2)
    x_left = chin[0] - half_width
    x_right = chin[0] + half_width
    
    # Draw vertical lines
    v_len = int(40 / mm_per_px)
    cv2.line(front_def, (x_left, y_line - v_len), (x_left, y_line + v_len), (255,255,255,200), 2)
    cv2.line(front_def, (x_right, y_line - v_len), (x_right, y_line + v_len), (255,255,255,200), 2)
    
    # Draw dotted horizontal line
    dist = x_right - x_left
    gap = 8
    for i in range(0, dist, gap*2):
        cv2.line(front_def, (x_left + i, y_line), (min(x_left + i + gap, x_right), y_line), (255,255,255,200), 1)
        
    def_b64 = img_to_b64(front_def)
    
    html_content = f"""
    <style>
        @import url('https://fonts.googleapis.com/css2?family=Inter:wght@300;400;500;600;700&display=swap');
        .d-dash {{ font-family: 'Inter', sans-serif; color: #2D3748; max-width: 1000px; margin: 0 auto; background: white; padding-bottom: 40px; }}
        .d-title {{ font-size: 32px; font-weight: 600; margin-bottom: 8px; letter-spacing: -0.5px; margin-top: 40px; }}
        .d-title span {{ color: #A0AEC0; }}
        .d-sub {{ font-size: 15px; color: #718096; margin-bottom: 32px; }}
        
        .d-grid {{ display: grid; grid-template-columns: 1fr 1fr; gap: 32px; }}
        
        .d-left {{ border: 1px solid #F1F5F9; border-radius: 16px; overflow: hidden; display: flex; align-items: center; justify-content: center; background: #F8FAFC; }}
        .d-left img {{ width: 100%; height: 100%; object-fit: cover; border-radius: 16px; }}
        
        .d-right {{ display: flex; flex-direction: column; gap: 20px; }}
        
        /* Score Panel */
        .score-panel {{ border: 1px solid #F1F5F9; border-radius: 16px; padding: 32px; text-align: center; display: flex; flex-direction: column; align-items: center; justify-content: center; position: relative; }}
        .sp-title {{ font-size: 16px; font-weight: 500; color: #1A202C; margin-bottom: 16px; }}
        .sp-score {{ font-size: 80px; font-weight: 400; color: #334155; line-height: 1; margin-bottom: 24px; letter-spacing: -2px; }}
        .sp-footer {{ display: flex; justify-content: space-between; width: 100%; align-items: flex-end; margin-top: auto; }}
        .sp-tag {{ background: #F0FDF4; color: #166534; padding: 4px 12px; border-radius: 12px; font-size: 12px; font-weight: 500; border: 1px solid #DCFCE7; }}
        .sp-max {{ font-size: 14px; color: #A0AEC0; font-weight: 500; }}
        
        /* Slider Panel */
        .slider-panel {{ border: 1px solid #F1F5F9; border-radius: 16px; padding: 24px; background: #F8FAFC; }}
        .sl-lbl {{ font-size: 11px; text-transform: uppercase; letter-spacing: 1.5px; color: #A0AEC0; font-weight: 600; margin-bottom: 8px; }}
        .sl-title {{ font-size: 20px; font-weight: 500; color: #1A202C; margin-bottom: 32px; }}
        .sl-track-wrap {{ position: relative; height: 16px; margin-bottom: 8px; }}
        .sl-track {{ height: 4px; background: #CBD5E1; border-radius: 2px; position: absolute; top: 50%; transform: translateY(-50%); width: 100%; }}
        .sl-thumb {{ width: 8px; height: 8px; background: #1A202C; border-radius: 2px; position: absolute; top: 50%; transform: translate(-50%, -50%); left: {muscularity_score}%; }}
        .sl-labels {{ display: flex; justify-content: space-between; font-size: 12px; font-weight: 600; color: #1A202C; }}
        
        /* Explanation Panel */
        .exp-panel {{ border: 1px solid #F1F5F9; border-radius: 16px; padding: 24px; background: #F8FAFC; flex-grow: 1; }}
        .exp-lbl {{ font-size: 11px; text-transform: uppercase; letter-spacing: 1.5px; color: #A0AEC0; font-weight: 600; margin-bottom: 16px; }}
        .exp-txt {{ font-size: 14px; color: #4A5568; line-height: 1.6; }}
    </style>
    
    <div class="d-dash">
        <div class="d-title">The definition of your <span>neck</span></div>
        <div class="d-sub">We evaluate your neck definition by assessing its <b>muscularity</b>, <b>vascularity</b>, and the visibility of its underlying structure.</div>
        
        <div class="d-grid">
            <div class="d-left">
                <img src="data:image/png;base64,{def_b64}" />
            </div>
            
            <div class="d-right">
                <div class="score-panel">
                    <div class="sp-title">Neck Muscularity</div>
                    <div class="sp-score">{muscularity_score}</div>
                    <div class="sp-footer">
                        <div class="sp-tag">{muscularity_tag}</div>
                        <div class="sp-max">/100</div>
                    </div>
                </div>
                
                <div class="slider-panel">
                    <div class="sl-lbl">NECK MUSCULARITY</div>
                    <div class="sl-title">{muscularity_cat}</div>
                    <div class="sl-track-wrap">
                        <div class="sl-track"></div>
                        <div class="sl-thumb"></div>
                    </div>
                    <div class="sl-labels">
                        <span>Slim</span>
                        <span>Muscular</span>
                    </div>
                </div>
                
                <div class="exp-panel">
                    <div class="exp-lbl">EXPLANATION</div>
                    <div class="exp-txt">{exp_text}</div>
                </div>
            </div>
        </div>
    </div>
    """
    
    from IPython.display import display, HTML
    display(HTML(html_content))

